# Pavement Distress Model Training and Evaluation

This notebook trains and evaluates models for predicting INDEX_PCR_2018 and RUT_AVG_2018, targeting accuracy (R² * 100) above 94%. It uses hyperparameter tuning, feature scaling, and refined binning to improve performance over previous models. Metrics include accuracy percentage, RMSE, MAE, F1 score, precision, recall, and confusion matrix for the original test set. Models are saved as pickle files for use in a Streamlit app. Verification on a new dataset reports only accuracy percentage and displays a table of actual vs. predicted values.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')
%matplotlib inline

In [2]:
# Load and preprocess data
def load_and_preprocess_data(file_path):
    try:
        df = pd.read_excel(file_path)
    except FileNotFoundError:
        print(f'Error: File {file_path} not found.')
        return None, None
    features = [
        'INDEX_IRI', 'IRI_AVG', 'INDEX_RUT', 'RUT_AVG', 'PERCENT_CRACKING',
        'ALLIG_H', 'ALLIG_M', 'LONG_H', 'LONG_M', 'TRAN_H', 'TRAN_M', 'SPEED', 'WIDTH'
    ]
    for col in features + ['INDEX_PCR_2018', 'RUT_AVG_2018']:
        if col in df.columns and df[col].isna().sum() > 0:
            df[col] = df[col].interpolate(method='linear', limit_direction='both')
    return df, features

# Load training data
df, features = load_and_preprocess_data('H0010k17_18_20_c.xlsx')
if df is None:
    raise SystemExit('Training data loading failed. Exiting.')
X = df[features]
y_pcr = df['INDEX_PCR_2018']
y_rut = df['RUT_AVG_2018']

# Split data
X_train, X_test, y_pcr_train, y_pcr_test = train_test_split(X, y_pcr, test_size=0.2, random_state=42)
X_train_rut, X_test_rut, y_rut_train, y_rut_test = train_test_split(X, y_rut, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_rut_scaled = scaler.fit_transform(X_train_rut)
X_test_rut_scaled = scaler.transform(X_test_rut)

In [ ]:
# Train models with hyperparameter tuning
def train_model(model, param_grid, X_train, y_train, model_name, target_name):
    print(f'Training {model_name} for {target_name}...')
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='r2', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    print(f'Best parameters for {model_name}: {grid_search.best_params_}')
    return grid_search.best_estimator_, grid_search.best_score_

# Define models and parameter grids
models = {
    'Random Forest': (RandomForestRegressor(random_state=42), {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }),
    'Gradient Boosting': (GradientBoostingRegressor(random_state=42), {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7],
        'min_samples_split': [2, 5]
    }),
    'SVM': (SVR(), {
        'C': [1, 10, 100],
        'epsilon': [0.01, 0.1, 0.2],
        'kernel': ['rbf', 'linear']
    }),
    'XGBoost': (XGBRegressor(random_state=42), {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7],
        'subsample': [0.8, 1.0]
    })
}

# Train models and store estimators and scores
pcr_models = {}
pcr_scores = {}
for name, (model, param_grid) in models.items():
    X_train_pcr = X_train_scaled if name == 'SVM' else X_train
    estimator, score = train_model(model, param_grid, X_train_pcr, y_pcr_train, name, 'INDEX_PCR_2018')
    pcr_models[name] = estimator
    pcr_scores[name] = max(score, 0)  

rut_models = {}
rut_scores = {}
for name, (model, param_grid) in models.items():
    X_train_rut = X_train_rut_scaled if name == 'SVM' else X_train_rut
    estimator, score = train_model(model, param_grid, X_train_rut, y_rut_train, name, 'RUT_AVG_2018')
    rut_models[name] = estimator
    rut_scores[name] = max(score, 0)  

# Save individual models
for name, model in pcr_models.items():
    with open(f'INDEX_PCR_2018_{name.lower().replace(" ", "_")}_model.pkl', 'wb') as f:
        pickle.dump(model, f)
for name, model in rut_models.items():
    with open(f'RUT_AVG_2018_{name.lower().replace(" ", "_")}_model.pkl', 'wb') as f:
        pickle.dump(model, f)
print('All individual models saved as pickle files.')

Training Random Forest for INDEX_PCR_2018...
Best parameters for Random Forest: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 300}
Training Gradient Boosting for INDEX_PCR_2018...
Best parameters for Gradient Boosting: {'learning_rate': 0.05, 'max_depth': 3, 'min_samples_split': 5, 'n_estimators': 200}
Training SVM for INDEX_PCR_2018...
Best parameters for SVM: {'C': 1, 'epsilon': 0.2, 'kernel': 'rbf'}
Training XGBoost for INDEX_PCR_2018...
Best parameters for XGBoost: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}
Training Random Forest for RUT_AVG_2018...
Best parameters for Random Forest: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Training Gradient Boosting for RUT_AVG_2018...
Best parameters for Gradient Boosting: {'learning_rate': 0.01, 'max_depth': 3, 'min_samples_split': 2, 'n_estimators': 300}
Training SVM for RUT_AVG_2018...
Best parameters for SVM: {'C': 1, 'epsilon':

In [4]:
# Select top two models based on R² scores
def select_top_models(scores, models, top_n=2):
    sorted_models = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    top_model_names = [name for name, _ in sorted_models]
    top_models = {name: models[name] for name in top_model_names}
    top_weights = {name: scores[name] for name in top_model_names}
    return top_models, top_weights

# Get top two models for PCR and RUT
top_pcr_models, top_pcr_weights = select_top_models(pcr_scores, pcr_models)
top_rut_models, top_rut_weights = select_top_models(rut_scores, rut_models)

# Define Ensemble Model class with weighted averaging for top two models
class EnsembleModel:
    def __init__(self, models, weights, scaler):
        self.models = models  # dict of name: model
        self.weights = weights  # dict of name: weight
        self.scaler = scaler

    def predict(self, X):
        predictions = {}
        for name, model in self.models.items():
            if name == 'SVM':
                X_scaled = self.scaler.transform(X)
                pred = model.predict(X_scaled)
            else:
                pred = model.predict(X)
            predictions[name] = pred
        total_weight = sum(self.weights.values())
        if total_weight == 0:  # Avoid division by zero
            return np.mean(list(predictions.values()), axis=0)
        ensemble_pred = sum(self.weights[name] * predictions[name] for name in self.models.keys()) / total_weight
        return ensemble_pred

# Create ensemble models with top two models
ensemble_pcr = EnsembleModel(top_pcr_models, top_pcr_weights, scaler)
ensemble_rut = EnsembleModel(top_rut_models, top_rut_weights, scaler)

# Save ensemble models
with open('ensemble_pcr_model.pkl', 'wb') as f:
    pickle.dump(ensemble_pcr, f)
with open('ensemble_rut_model.pkl', 'wb') as f:
    pickle.dump(ensemble_rut, f)
print('Ensemble models saved as pickle files.')

Ensemble models saved as pickle files.


In [5]:
# Function to bin continuous values into classes for original test set
def bin_predictions(y_true, y_pred, target_name):
    if target_name == 'INDEX_PCR_2018':
        bins = [0, 50, 70, 85, 100]
        labels = [0, 1, 2, 3]
    else:
        bins = [0, 0.15, 0.3, 0.5, np.inf]
        labels = [0, 1, 2, 3]
    y_true_binned = pd.cut(y_true, bins=bins, labels=labels, include_lowest=True)
    y_pred_binned = pd.cut(y_pred, bins=bins, labels=labels, include_lowest=True)
    return y_true_binned, y_pred_binned

In [11]:
# Evaluate models on original test set
def evaluate_model(model, X_test, y_test, target_name, model_name):
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    accuracy_percentage = r2 * 100
    y_true_binned, y_pred_binned = bin_predictions(y_test, y_pred, target_name)
    precision = precision_score(y_true_binned, y_pred_binned, average='weighted', zero_division=0)
    recall = recall_score(y_true_binned, y_pred_binned, average='weighted', zero_division=0)
    f1 = f1_score(y_true_binned, y_pred_binned, average='weighted', zero_division=0)
    cm = confusion_matrix(y_true_binned, y_pred_binned)
    print(f'\n{model_name} for {target_name}:')
    print(f'RMSE: {rmse:.4f}')
    print(f'MAE: {mae:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1 Score: {f1:.4f}')
    
    return {
        'RMSE': rmse,
        'MAE': mae,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    }

In [12]:
# Evaluate ensemble models on original test set
print('\nEvaluation on Original Test Set:')
print('\nEnsemble (Top 2) for INDEX_PCR_2018:')
evaluate_model(ensemble_pcr, X_test, y_pcr_test, 'INDEX_PCR_2018', 'Ensemble (Top 2)')
print('\nEnsemble (Top 2) for RUT_AVG_2018:')
evaluate_model(ensemble_rut, X_test_rut, y_rut_test, 'RUT_AVG_2018', 'Ensemble (Top 2)')


Evaluation on Original Test Set:

Ensemble (Top 2) for INDEX_PCR_2018:

Ensemble (Top 2) for INDEX_PCR_2018:
RMSE: 0.5172
MAE: 0.3803
Precision: 1.0000
Recall: 1.0000
F1 Score: 1.0000

Ensemble (Top 2) for RUT_AVG_2018:

Ensemble (Top 2) for RUT_AVG_2018:
RMSE: 0.1037
MAE: 0.0789
Precision: 0.5581
Recall: 0.5759
F1 Score: 0.5026


{'RMSE': 0.10368807500915372,
 'MAE': 0.07890480919031967,
 'Precision': 0.5580560803801057,
 'Recall': 0.5759036144578313,
 'F1 Score': 0.5026088522016917}

In [1]:
# Function to verify model on new dataset
def verify_model_on_new_dataset(model, scaler, new_data_path, target_name, model_name, features):
    if not os.path.exists(new_data_path):
        print(f'Error: Dataset {new_data_path} does not exist. Please provide a valid file path.')
        return None
    new_df, _ = load_and_preprocess_data(new_data_path)
    if new_df is None:
        return None
    
    # Ensure new dataset has required columns
    if not all(col in new_df.columns for col in features + [target_name]):
        missing_cols = [col for col in features + [target_name] if col not in new_df.columns]
        print(f'Error: Missing columns in new dataset: {missing_cols}')
        return None
    
    X_new = new_df[features]
    y_new = new_df[target_name]
    
    y_pred = model.predict(X_new)
    r2 = r2_score(y_new, y_pred)
    accuracy_percentage = r2 * 100
    
    print(f'\nVerification Results for {model_name} on {target_name} (New Dataset):')
    print(f'Accuracy (R² * 100): {accuracy_percentage:.2f}%')
    
    results_df = pd.DataFrame({'Actual': y_new, 'Predicted': y_pred})
    print('\nActual vs. Predicted Values:')
    print(results_df.head(10))
    
    return {'Accuracy (%)': accuracy_percentage}

# Specify new dataset path
new_data_path = 'H0010k17_18_20_c.xlsx'  
new_data_pathl = 'H0020k17_18_20_l.xlsx'
# # Verify ensemble models on new dataset
# print('\nVerification on New Dataset:')
# print('\nVerifying Ensemble for INDEX_PCR_2018 on new dataset...')
# verify_model_on_new_dataset(ensemble_pcr, scaler, new_data_path, 'INDEX_PCR_2018', 'Ensemble (Top 2)', features)
# print('\nVerifying Ensemble for RUT_AVG_2018 on new dataset...')
# verify_model_on_new_dataset(ensemble_rut, scaler, new_data_path, 'RUT_AVG_2018', 'Ensemble (Top 2)', features)

In [41]:
new_data_path1 = 'H0020k17_18_20_1.xlsx'  
print('\nVerification on New Dataset:')

print('\nVerifying Ensemble (Top 2) for INDEX_PCR_2018 on new dataset...')
verify_model_on_new_dataset(ensemble_pcr, scaler, new_data_path1, 'INDEX_PCR_2018', 'Ensemble (Top 2)', features)
print('\nVerifying Ensemble (Top 2) for RUT_AVG_2018 on new dataset...')
verify_model_on_new_dataset(ensemble_rut, scaler, new_data_pathl, 'RUT_AVG_2018', 'Ensemble (Top 2)', features)


Verification on New Dataset:

Verifying Ensemble (Top 2) for INDEX_PCR_2018 on new dataset...

Verification Results for Ensemble (Top 2) on INDEX_PCR_2018 (New Dataset):
Accuracy (R² * 100): 90.17%

Actual vs. Predicted Values:
     Actual  Predicted
0  3.977942   4.008869
1  4.109615   3.971155
2  4.056833   3.923507
3  3.875882   3.964339
4  3.707292   3.582359
5  3.632046   3.487584
6  3.556800   3.512277
7  2.954830   3.283273
8  2.879584   3.191690
9  3.804518   3.704772

Verifying Ensemble (Top 2) for RUT_AVG_2018 on new dataset...

Verification Results for Ensemble (Top 2) on RUT_AVG_2018 (New Dataset):
Accuracy (R² * 100): 92.64%

Actual vs. Predicted Values:
     Actual  Predicted
0  0.281514   0.282513
1  0.213486   0.219998
2  0.300908   0.312858
3  0.314114   0.308145
4  0.297108   0.308544
5  0.272941   0.271704
6  0.280972   0.286860
7  0.233092   0.218168
8  0.270853   0.271553


{'Accuracy (%)': 92.64270522015923}